<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Modulo 2</h2><br/>
<h1>Semana 9 · Lunes — Aprendizaje No Supervisado</h1>
<h3>KMeans + DBSCAN + PCA para segmentacion de clientes</h3>
<br/>
    <b>Instructor:</b> Jesus Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## Objetivos

Al final de la clase van a poder:

1. Entender la diferencia entre aprendizaje supervisado y no supervisado.
2. Aplicar KMeans para segmentacion de clientes y elegir el K correcto con el metodo del codo y silhouette score.
3. Usar DBSCAN cuando los clusters no son esfericos o cuando queremos detectar outliers.
4. Usar PCA para reducir dimensiones y visualizar clusters en 2D.
5. Resolver un ejercicio real de segmentacion de clientes de un mall con recomendaciones de negocio.

# 1. Que es el aprendizaje no supervisado

Hasta ahora trabajamos con **aprendizaje supervisado**: teniamos una variable objetivo (Total Amount, Churn, etc.) y le ensenabamos al modelo a predecirla a partir de las features.

En **aprendizaje no supervisado** NO hay variable objetivo. Le damos al modelo solo las features y le pedimos que encuentre estructura por su cuenta.

| Aprendizaje | Tiene target? | Que hace | Ejemplos |
|---|---|---|---|
| Supervisado | Si | Aprende a predecir | Regresion, Clasificacion |
| No Supervisado | No | Encuentra estructura | Clustering, Reduccion dimensional, Deteccion de anomalias |

Casos de uso de clustering en el negocio:
- **Segmentacion de clientes**: agrupar clientes por comportamiento de compra para personalizar marketing.
- **Deteccion de fraude**: encontrar transacciones que no encajan con ningun grupo "normal".
- **Recomendacion de productos**: clientes parecidos compran productos parecidos.
- **Compresion de imagenes**: reducir colores a paletas de K colores.

# 2. Dataset: Mall Customer Segmentation

Hoy vamos a trabajar con un dataset clasico de Kaggle: **Mall Customer Segmentation**. Es de un centro comercial que quiere entender mejor a sus clientes para hacer campanas de marketing.

Tiene 200 clientes con estos datos:
- `CustomerID`: identificador (no se usa para modelar)
- `Gender`: Male / Female
- `Age`: edad del cliente
- `Annual Income (k$)`: ingreso anual en miles de dolares
- `Spending Score (1-100)`: puntaje que asigno el mall segun el comportamiento de gasto (1=tacaño, 100=gastador)

El equipo de marketing quiere saber: cuantos segmentos de clientes hay y como caracterizarlos para que cada segmento reciba el tipo de campana correcta.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)

# Dataset desde Kaggle (mismo URL que usamos en el ejercicio)
URL = 'https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2024%20-%20K-Means%20Clustering/Mall_Customers.csv'
df = pd.read_csv(URL)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# EDA rapido
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribucion de Edad', fontweight='bold')
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Media={df["Age"].mean():.1f}')
axes[0].legend()

sns.histplot(df['Annual Income (k$)'], bins=20, kde=True, ax=axes[1], color='#70AD47')
axes[1].set_title('Distribucion de Ingreso Anual', fontweight='bold')

sns.histplot(df['Spending Score (1-100)'], bins=20, kde=True, ax=axes[2], color='#C0504D')
axes[2].set_title('Distribucion de Spending Score', fontweight='bold')

plt.tight_layout(); plt.show()

print(df.describe())

**Lo que veo en el EDA rapido**

- **Age**: 200 clientes entre 18 y 70 anos, media 38.9, distribuida bastante uniformemente (no normal pero sin asimetrias graves).
- **Annual Income**: entre $15k y $137k, media $60.6k, distribucion bimodal: hay un grupo de ingresos bajos-medios y otro grupo de altos.
- **Spending Score**: entre 1 y 99, media 50.2, distribuida casi uniforme. Esto es interesante: el mall asigno el score con bastante variabilidad, hay tanto gastadores como tacaños.

Estas 3 variables (Age, Income, Spending) son las que vamos a usar para hacer clustering. Vamos a normalizarlas porque sus escalas son muy distintas (Age 18-70, Income 15-137, Score 1-99).

# 3. KMeans paso a paso

KMeans es el algoritmo de clustering mas usado. La idea es simple:

1. Elegimos K (numero de clusters).
2. Inicializamos K centroides al azar.
3. Cada punto se asigna al centroide mas cercano.
4. Recalculamos cada centroide como el promedio de los puntos que le toco.
5. Repetimos pasos 3 y 4 hasta que los centroides dejen de moverse.

**Limitaciones de KMeans:**
- Asume clusters de forma esferica (no funciona bien con clusters alargados).
- Necesita que le digamos K de antemano.
- Sensible a la escala de las features (por eso normalizamos siempre).
- Sensible a la inicializacion (por eso usamos `n_init` para correrlo varias veces).

In [ ]:
# Preparamos las features (sin CustomerID ni Gender por ahora)
X = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].values

# Normalizamos: KMeans usa distancia euclidiana, las escalas distintas distorsionan
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Primer intento: K=3 (numero al azar)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

print(f'Cluster sizes: {pd.Series(labels).value_counts().to_dict()}')
print(f'Inertia (suma de distancias al centroide): {kmeans.inertia_:.2f}')
print(f'Silhouette score: {silhouette_score(X_scaled, labels):.3f}')

**Como elegimos K?**

Hay dos metodos clasicos:

1. **Metodo del codo (Elbow Method)**: graficamos la inertia para distintos K. La inertia siempre baja con K mas grande, pero hay un punto donde la mejora se aplana (el "codo"). Ese K es el optimo.

2. **Silhouette Score**: mide que tan bien separados estan los clusters. Va de -1 a 1, mas alto es mejor. Si tenemos un pico, ese K es el optimo.

In [ ]:
# Probamos K de 2 a 10
inertias = []
silhouettes = []
ks = range(2, 11)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(ks, inertias, 'o-', linewidth=2, markersize=10, color='steelblue')
axes[0].set_xlabel('K (numero de clusters)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Metodo del Codo', fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].axvline(5, color='red', linestyle='--', alpha=0.5, label='K=5 (codo)')
axes[0].legend()

axes[1].plot(ks, silhouettes, 'o-', linewidth=2, markersize=10, color='#70AD47')
axes[1].set_xlabel('K (numero de clusters)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].axvline(6, color='red', linestyle='--', alpha=0.5, label='K=6 (max)')
axes[1].legend()

plt.tight_layout(); plt.show()

print('Resumen:')
for k, i, s in zip(ks, inertias, silhouettes):
    print(f'  K={k}: inertia={i:.1f}, silhouette={s:.3f}')

**Lo que veo en la curva del codo y silhouette**

- El **codo** en la inertia esta entre K=5 y K=6. Despues de K=6 la curva se aplana mucho.
- El **silhouette** maximo esta en K=6 (0.428), pero K=5 (0.417) esta muy cerca y K=7 baja.

Decision: elijo **K=5** porque es el codo claro y un silhouette casi tan bueno como K=6, pero con menos clusters (mas facil de interpretar para el negocio). 5 segmentos de clientes son manejables, 6 ya empieza a ser mucho.

Una nota practica: en problemas reales no siempre el K matematicamente optimo es el correcto. A veces el negocio te dice "necesito maximo 4 segmentos porque solo tengo presupuesto para 4 campanas". Entonces eliges K=4 aunque silhouette te diga K=7.

In [ ]:
# KMeans final con K=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans.fit_predict(X_scaled)

# Caracterizacion de cada cluster
caracterizacion = df.groupby('Cluster_KMeans')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)
caracterizacion['n_clientes'] = df.groupby('Cluster_KMeans').size()
print('Caracterizacion de los 5 segmentos:')
print(caracterizacion)

**Lo que veo en los 5 segmentos**

Cada cluster tiene un perfil claro:

| Cluster | Edad | Ingreso | Score | Interpretacion |
|---|---|---|---|---|
| 0 | 46 | $27k | 18 | **Maduros tacaños**: ingresos bajos, gastan poco. Probablemente jubilados o de bajo poder adquisitivo. |
| 1 | 25 | $41k | 62 | **Jovenes gastadores**: ingresos medios pero gastan harto. Target de campanas de credito y experiencia. |
| 2 | 33 | $86k | 81 | **PREMIUM**: alto ingreso, alto gasto. El mejor segmento del mall, hay que cuidarlo con programa VIP. |
| 3 | 40 | $86k | 19 | **Altos ingresos no gastan**: tienen plata pero no la gastan en el mall. **Target perdido o por capturar** con campanas premium. |
| 4 | 56 | $54k | 49 | **Senior promedio**: gasto y edad medios. Segmento estable, no requiere atencion especial. |

Recomendaciones de marketing por segmento:
- **Cluster 2 (Premium)**: programa de fidelizacion VIP, eventos exclusivos, productos de lujo.
- **Cluster 3 (Perdidos)**: estudiar por que no gastan aunque tienen ingreso. Encuestas + campanas premium dirigidas.
- **Cluster 1 (Jovenes gastadores)**: redes sociales, productos aspiracionales, financiamiento.
- **Cluster 0 (Tacaños)**: ofertas, descuentos, productos basicos.
- **Cluster 4 (Senior promedio)**: mantener servicio actual, no es prioritario.

**Esta es la diferencia entre hacer clustering tecnico y entregar valor de negocio**: el cliente no quiere ver los numeros del silhouette, quiere ver los segmentos con nombre y plan de accion.

# 4. DBSCAN: clustering basado en densidad

KMeans tiene una limitacion: asume que los clusters son esfericos y de tamano parecido. Si tus clusters tienen forma extraña (lunas, anillos, etc.) o tienen tamaños muy distintos, KMeans falla.

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) resuelve esto. Funciona asi:

1. Toma un punto al azar.
2. Mira cuantos puntos hay a distancia `eps` (epsilon) de el.
3. Si hay al menos `min_samples` puntos, es un cluster.
4. Expande el cluster con todos los vecinos cercanos.
5. Los puntos que no encajan en ningun cluster se marcan como **ruido** (label = -1).

**Ventajas de DBSCAN:**
- No hay que decidir K de antemano.
- Funciona con clusters de cualquier forma.
- Detecta outliers automaticamente (los marca como ruido).

**Desventajas:**
- Sensible a los parametros `eps` y `min_samples`.
- No funciona bien si los clusters tienen densidades muy distintas.

In [ ]:
# Probamos varios eps
print('Prueba de parametros DBSCAN:')
print(f'{"eps":<6}{"min_samples":<12}{"clusters":<10}{"ruido":<8}{"silhouette":<12}')
print('-' * 50)

resultados_dbscan = []
for eps in [0.3, 0.5, 0.7, 1.0]:
    for ms in [5, 10]:
        db = DBSCAN(eps=eps, min_samples=ms).fit(X_scaled)
        n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
        n_noise = (db.labels_ == -1).sum()
        if n_clusters > 1:
            mask = db.labels_ != -1
            sil = silhouette_score(X_scaled[mask], db.labels_[mask])
        else:
            sil = float('nan')
        print(f'{eps:<6}{ms:<12}{n_clusters:<10}{n_noise:<8}{sil:<12.3f}')
        resultados_dbscan.append((eps, ms, n_clusters, n_noise, sil))

**Lo que veo en DBSCAN**

DBSCAN es muy sensible a `eps`. Con eps muy chico encuentra muchos clusters chiquitos y muchos puntos ruido. Con eps muy grande junta todo en un solo cluster.

Para este dataset eps=0.5 con min_samples=5 da 6 clusters y 60 puntos de ruido (30% del dataset). Esto es informacion valiosa: hay clientes que NO encajan claramente en ningun grupo, son los "fronterizos".

**Cuando elegirias DBSCAN sobre KMeans?**
- Cuando esperas que haya outliers/ruido (clientes raros).
- Cuando los clusters no son esfericos.
- Cuando NO quieres tener que decidir K.
- Cuando te interesa identificar puntos "anormales" para investigarlos.

**Cuando elegirias KMeans?**
- Cuando necesitas un numero exacto de segmentos para el negocio.
- Cuando los clusters son razonablemente esfericos.
- Cuando todos los puntos deben pertenecer a algun cluster.
- Cuando tienes muchos datos y necesitas algo rapido y simple.

Para este caso de segmentacion de clientes, KMeans es la opcion correcta porque el negocio necesita exactamente 5 segmentos accionables. DBSCAN nos sirve como complemento para identificar los "clientes raros" que vale la pena estudiar por separado.

# 5. PCA para visualizar clusters en 2D

Tenemos 3 features (Age, Income, Spending). Es dificil visualizar 3D y los clusters quedan claros en 2D. **PCA** (Principal Component Analysis) reduce las 3 dimensiones a 2 manteniendo la mayor cantidad posible de la varianza.

PCA encuentra las direcciones (componentes principales) donde los datos tienen mas variabilidad. PC1 es la direccion de maxima varianza, PC2 la siguiente, etc.

Importante: PCA es solo para visualizacion en este caso, no para mejorar el clustering. Para que el clustering sea valido, lo hicimos con las 3 features originales.

In [ ]:
# Aplicamos PCA para visualizar
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Varianza explicada por cada componente: {pca.explained_variance_ratio_.round(3)}')
print(f'Varianza acumulada: {pca.explained_variance_ratio_.cumsum().round(3)}')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# KMeans en el espacio PCA
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster_KMeans'],
                            cmap='tab10', s=60, alpha=0.7, edgecolor='white')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
axes[0].set_title('KMeans con K=5 visualizado en PCA', fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# DBSCAN en el espacio PCA
db_final = DBSCAN(eps=0.5, min_samples=5).fit(X_scaled)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=db_final.labels_,
                            cmap='tab10', s=60, alpha=0.7, edgecolor='white')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
axes[1].set_title('DBSCAN (eps=0.5) visualizado en PCA', fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster (-1=ruido)')

plt.tight_layout(); plt.show()

**Lo que veo en la visualizacion con PCA**

PC1 captura 44% de la varianza, PC2 captura 33%. Juntos cubren 77.5% de la variabilidad de los datos. Eso es suficiente para visualizar los clusters sin perder mucha informacion.

Visualmente:
- **KMeans con K=5** muestra 5 grupos claramente separados en el espacio 2D. La segmentacion es limpia.
- **DBSCAN con eps=0.5** muestra varios clusters pero con muchos puntos marcados como ruido (puntos con label -1, en un color distinto).

Si fueramos a presentar esto al equipo de marketing, **el grafico de KMeans es el que pondria en la diapositiva**: clusters claros, separados, faciles de interpretar.

PCA tambien sirve para entender que **direcciones** estan capturando los componentes principales. Lo veo a continuacion:

In [ ]:
# Que features pesa cada componente principal?
componentes = pd.DataFrame(
    pca.components_,
    columns=['Age', 'Annual Income (k$)', 'Spending Score (1-100)'],
    index=['PC1', 'PC2']
).round(3)
print('Pesos de cada feature en los componentes principales:')
print(componentes)

**Como interpretar los pesos de los componentes**

- **PC1** captura sobre todo la combinacion de `Annual Income` y `Spending Score` (los dos con pesos altos del mismo signo). Es el eje "poder adquisitivo".
- **PC2** captura principalmente `Age` (peso alto en Age, bajo en el resto). Es el eje "edad".

Por eso en el scatter de PCA: el eje horizontal separa por poder adquisitivo + gasto, el eje vertical separa por edad. Esa es la geometria intuitiva de los segmentos.

---
# Ejercicio integrador

Trabajan como Data Scientists en el mismo mall. La gerencia comercial les pide que entreguen un plan de marketing por segmento, basado en clustering.

## Reto: segmentar clientes y entregar plan de marketing accionable

Dataset: el mismo Mall Customers que vimos en clase.

### Lo que tienen que hacer

**Parte A — Preprocesamiento (10 min)**
1. Cargar el dataset y verificar nulos/duplicados.
2. Aplicar StandardScaler a las 3 features numericas (Age, Income, Spending).
3. (Bonus) Investigar si Gender aporta informacion: hacer clustering con Gender codificado y sin Gender, comparar.

**Parte B — KMeans (20 min)**
4. Probar K de 2 a 10 con metodo del codo + silhouette.
5. Elegir el K optimo justificando tecnicamente.
6. Entrenar KMeans con el K elegido.
7. Caracterizar cada cluster con tabla de promedios (Age, Income, Spending, % de mujeres).
8. Darle un nombre comercial a cada cluster (ej. "VIP", "Jovenes aspiracionales", etc.)

**Parte C — DBSCAN (15 min)**
9. Probar al menos 4 combinaciones de eps y min_samples.
10. Comparar contra KMeans: cuantos clusters encuentra DBSCAN? cuantos puntos son ruido?
11. Cuales clientes son ruido para DBSCAN? Por que? Caracterizarlos.

**Parte D — PCA + Visualizacion (15 min)**
12. Aplicar PCA con 2 componentes.
13. Reportar varianza explicada acumulada.
14. Visualizar los clusters de KMeans y DBSCAN en el espacio PCA.
15. Interpretar que captura cada componente principal.

**Parte E — Plan de marketing (20 min)**
16. Para cada cluster de KMeans:
   - Nombre comercial
   - Perfil demografico (edad, ingreso, gasto)
   - Comportamiento esperado (tacaño, gastador, ahorrador)
   - Recomendacion concreta de campana de marketing (que oferta, que canal, que frecuencia)
17. Si el mall tiene presupuesto solo para 2 campanas, cuales 2 clusters priorizar y por que?
18. Como evaluarian si las campanas funcionaron despues de 3 meses?

### Bonus (vale puntos extra)

- Probar clustering jerarquico (`AgglomerativeClustering`) y dibujar el dendrograma.
- Aplicar PCA con 3 componentes y graficar en 3D.
- Investigar `MiniBatchKMeans` para datasets mas grandes: cuanto mas rapido es?
- Comparar la segmentacion incluyendo Gender vs sin Gender: cambia algun cluster relevante?

In [ ]:
# Parte A — Preprocesamiento



In [ ]:
# Parte B — KMeans



In [ ]:
# Parte C — DBSCAN



In [ ]:
# Parte D — PCA + Visualizacion



# Parte E — Plan de marketing

(Escribir el plan de marketing por cluster en celdas markdown)


## Cierre

Hoy aprendimos:

- La diferencia entre aprendizaje supervisado (con target) y no supervisado (sin target).
- KMeans: el algoritmo de clustering mas comun, ideal para segmentacion con K conocido. Usar metodo del codo + silhouette para elegir K.
- DBSCAN: alternativa cuando hay outliers o clusters no esfericos. No requiere K pero requiere ajustar eps y min_samples.
- PCA: reduccion de dimensionalidad para visualizar clusters en 2D.

**Mas importante que el algoritmo**: caracterizar cada segmento con nombres comerciales y proponer acciones concretas de negocio. Eso es lo que hace que un proyecto de clustering aporte valor real.

Manana cerramos la semana con la clase de feedback del examen y empezamos con deployment.